In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/Crawler")
DATA_DIR = BASE_DIR / "data"

#BASE_DIR.mkdir(parents=True, exist_ok=True)
#DATA_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)

BASE_DIR: /content/drive/MyDrive/Crawler
DATA_DIR: /content/drive/MyDrive/Crawler/data


In [ ]:
from getpass import getpass

env_path = BASE_DIR / ".env"

# kakao_key = getpass("Kakao REST API Key 입력: ").strip()

# env_path.write_text(
#     f"KAKAO_REST_API_KEY={kakao_key}\n",
#     encoding="utf-8"
# )

# print(".env 생성 완료:", env_path)

In [ ]:
print(env_path.exists())
print(env_path.read_text(encoding="utf-8")[:30] + "...")

True
KAKAO_REST_API_KEY=4a1bda9139c...


In [ ]:
a= env_path.read_text(encoding="utf-8")

In [ ]:
!pip -q install selenium webdriver-manager pandas requests python-dotenv tqdm beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 10.5 MB/s eta 0:00:00


In [ ]:
gitignore_path = BASE_DIR / ".gitignore"

gitignore_path.write_text(
    """.env
__pycache__/
.ipynb_checkpoints/
*.log
data/
""",
    encoding="utf-8"
)

print(".gitignore 생성 완료:", gitignore_path)
print(gitignore_path.read_text(encoding="utf-8"))

.gitignore 생성 완료: /content/drive/MyDrive/Crawler/.gitignore
.env
__pycache__/
.ipynb_checkpoints/
*.log
data/



In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(env_path)

kakao_key = a[19:].strip()

print("키 존재 여부:", kakao_key is not None)
print("키 앞부분:", kakao_key[:6] + "..." if kakao_key else None)

키 존재 여부: True
키 앞부분: 4a1bda...


In [ ]:
import requests

headers = {
    "Authorization": f"KakaoAK {kakao_key}"
}

url = "https://dapi.kakao.com/v2/local/search/keyword.json"
params = {
    "query": "성균관대학교 자연과학캠퍼스 후문",
    "size": 5
}

res = requests.get(url, headers=headers, params=params, timeout=10)

print("status:", res.status_code)
print(res.json())

status: 200
{'documents': [{'address_name': '경기 수원시 장안구 천천동 300', 'category_group_code': '', 'category_group_name': '', 'category_name': '교통,수송 > 입출구', 'distance': '', 'id': '24725711', 'phone': '', 'place_name': '성균관대학교 자연과학캠퍼스 후문', 'place_url': 'http://place.map.kakao.com/24725711', 'road_address_name': '경기 수원시 장안구 서부로 2066', 'x': '126.970599473947', 'y': '37.2963102729076'}, {'address_name': '경기 수원시 장안구 천천동 300', 'category_group_code': '', 'category_group_name': '', 'category_name': '교통,수송 > 입출구', 'distance': '', 'id': '27435180', 'phone': '', 'place_name': '성균관대학교 자연과학캠퍼스 북문', 'place_url': 'http://place.map.kakao.com/27435180', 'road_address_name': '경기 수원시 장안구 서부로 2066', 'x': '126.976387573969', 'y': '37.2962250695826'}], 'meta': {'is_end': True, 'pageable_count': 2, 'same_name': {'keyword': '성균관대학교 자연과학캠퍼스 후문', 'region': [], 'selected_region': ''}, 'total_count': 2}}


In [ ]:
import math
import time
import pandas as pd

In [ ]:
CENTER_QUERY = "성균관대학교 자연과학캠퍼스 후문"
RADIUS_M = 300
CATEGORY_GROUP_CODE = "FD6"  # 음식점

headers = {
    "Authorization": f"KakaoAK {kakao_key.strip()}"
}

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)

BASE_DIR: /content/drive/MyDrive/Crawler
DATA_DIR: /content/drive/MyDrive/Crawler/data


In [ ]:
url = "https://dapi.kakao.com/v2/local/search/keyword.json"
params = {
    "query": CENTER_QUERY,
    "size": 5
}

res = requests.get(url, headers=headers, params=params, timeout=10)
print("status:", res.status_code)

data = res.json()
docs = data.get("documents", [])

if not docs:
    raise RuntimeError("후문 중심 좌표를 찾지 못했습니다.")

center = docs[0]

CENTER_LON = float(center["x"])
CENTER_LAT = float(center["y"])

print("중심 장소:", center["place_name"])
print("주소:", center["address_name"])
print("도로명:", center["road_address_name"])
print("중심 좌표:", CENTER_LON, CENTER_LAT)

status: 200
중심 장소: 성균관대학교 자연과학캠퍼스 후문
주소: 경기 수원시 장안구 천천동 300
도로명: 경기 수원시 장안구 서부로 2066
중심 좌표: 126.970599473947 37.2963102729076


In [ ]:
def haversine_m(lon1, lat1, lon2, lat2):
    r = 6371000
    p1 = math.radians(lat1)
    p2 = math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)

    a = (
        math.sin(dp / 2) ** 2
        + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    )
    return 2 * r * math.asin(math.sqrt(a))


def collect_kakao_restaurant_candidates(center_lon, center_lat, radius_m=300):
    url = "https://dapi.kakao.com/v2/local/search/category.json"
    rows = []

    for page in range(1, 46):
        params = {
            "category_group_code": CATEGORY_GROUP_CODE,
            "x": center_lon,
            "y": center_lat,
            "radius": radius_m,
            "sort": "distance",
            "page": page,
            "size": 15,
        }

        res = requests.get(url, headers=headers, params=params, timeout=10)
        print(f"page={page}, status={res.status_code}")

        if res.status_code != 200:
            print(res.text)
            break

        data = res.json()
        documents = data.get("documents", [])

        for d in documents:
            lon = float(d["x"])
            lat = float(d["y"])

            distance_m = haversine_m(center_lon, center_lat, lon, lat)

            if distance_m <= radius_m:
                rows.append({
                    "kakao_place_id": d.get("id"),
                    "name": d.get("place_name"),
                    "category_name": d.get("category_name"),
                    "address": d.get("address_name"),
                    "road_address": d.get("road_address_name"),
                    "longitude": lon,
                    "latitude": lat,
                    "phone": d.get("phone"),
                    "kakao_place_url": d.get("place_url"),
                    "distance_m": round(distance_m, 1),
                })

        if data.get("meta", {}).get("is_end"):
            break

        time.sleep(0.2)

    df = pd.DataFrame(rows)

    if df.empty:
        return df

    df = df.drop_duplicates(subset=["kakao_place_id"])
    df = df.sort_values(["distance_m", "name"]).reset_index(drop=True)

    df.insert(0, "restaurant_temp_id", range(1, len(df) + 1))
    df.insert(1, "campus_area", "skku_natural_back_gate")
    df.insert(2, "center_query", CENTER_QUERY)
    df.insert(3, "radius_m", radius_m)

    return df

In [ ]:
df_candidates = collect_kakao_restaurant_candidates(
    center_lon=CENTER_LON,
    center_lat=CENTER_LAT,
    radius_m=RADIUS_M
)

print("수집된 음식점 후보 수:", len(df_candidates))

display_cols = [
    "restaurant_temp_id",
    "name",
    "category_name",
    "address",
    "road_address",
    "distance_m",
    "kakao_place_url",
]

display(df_candidates[display_cols].head(50))

out_path = DATA_DIR / "01_kakao_restaurant_candidates.csv"
df_candidates.to_csv(out_path, index=False, encoding="utf-8-sig")

print("저장 완료:", out_path)

page=1, status=200
page=2, status=200
page=3, status=200
수집된 음식점 후보 수: 45


,restaurant_temp_id,name,category_name,address,road_address,distance_m,kakao_place_url
0,1,낭만가득한20세기포장마차 수원성대본점,음식점 > 술집 > 실내포장마차,경기 수원시 장안구 율전동 419,경기 수원시 장안구 서부로 2095,66.2,http://place.map.kakao.com/824736542
1,2,소리산전집,음식점 > 한식,경기 수원시 장안구 율전동 433-79,경기 수원시 장안구 서부로 2107,68.2,http://place.map.kakao.com/306868449
2,3,두루정 수원성대점,음식점 > 한식,경기 수원시 장안구 율전동 433-79,경기 수원시 장안구 서부로 2107,68.7,http://place.map.kakao.com/1279255339
3,4,코아김밥,음식점 > 분식,경기 수원시 장안구 율전동 419,경기 수원시 장안구 서부로 2095,71.4,http://place.map.kakao.com/762026624
4,5,김치로우주정복 수원율천점,"음식점 > 한식 > 찌개,전골 > 김치로우주정복",경기 수원시 장안구 율전동 419,경기 수원시 장안구 서부로 2095,71.8,http://place.map.kakao.com/1138868621
5,6,최고집 원조 수구레,음식점 > 한식,경기 수원시 장안구 율전동 433-79,경기 수원시 장안구 서부로 2107,72.1,http://place.map.kakao.com/1676208114
6,7,돼지게티 율천점,음식점 > 퓨전요리 > 돼지게티,경기 수원시 장안구 율전동 419,경기 수원시 장안구 서부로 2095,72.3,http://place.map.kakao.com/1687006483
7,8,미,음식점 > 술집,경기 수원시 장안구 율전동 433-79,경기 수원시 장안구 서부로 2107,74.2,http://place.map.kakao.com/382531552
8,9,신전떡볶이 수원성대점,음식점 > 분식 > 떡볶이 > 신전떡볶이,경기 수원시 장안구 율전동 433-79,경기 수원시 장안구 서부로 2107,76.3,http://place.map.kakao.com/24583309
9,10,중경마라탕 3호점,음식점 > 중식 > 중국요리,경기 수원시 장안구 율전동 433-68,경기 수원시 장안구 서부로2106번길 10,77.6,http://place.map.kakao.com/2064565238


저장 완료: /content/drive/MyDrive/Crawler/data/01_kakao_restaurant_candidates.csv


In [ ]:
out_path = DATA_DIR / "01_kakao_restaurant_candidates.csv"

In [ ]:
print(out_path)
print(out_path.exists())

df_check = pd.read_csv(out_path)
print(df_check.shape)
display(df_check.head())

/content/drive/MyDrive/Crawler/data/01_kakao_restaurant_candidates.csv
True
(45, 14)


,restaurant_temp_id,campus_area,center_query,radius_m,kakao_place_id,name,category_name,address,road_address,longitude,latitude,phone,kakao_place_url,distance_m
0,1,skku_natural_back_gate,성균관대학교 자연과학캠퍼스 후문,300,824736542,낭만가득한20세기포장마차 수원성대본점,음식점 > 술집 > 실내포장마차,경기 수원시 장안구 율전동 419,경기 수원시 장안구 서부로 2095,126.969930,37.296044,NaN,http://place.map.kakao.com/824736542,66.2
1,2,skku_natural_back_gate,성균관대학교 자연과학캠퍼스 후문,300,306868449,소리산전집,음식점 > 한식,경기 수원시 장안구 율전동 433-79,경기 수원시 장안구 서부로 2107,126.970242,37.296854,NaN,http://place.map.kakao.com/306868449,68.2
2,3,skku_natural_back_gate,성균관대학교 자연과학캠퍼스 후문,300,1279255339,두루정 수원성대점,음식점 > 한식,경기 수원시 장안구 율전동 433-79,경기 수원시 장안구 서부로 2107,126.970118,37.296795,031-293-2007,http://place.map.kakao.com/1279255339,68.7
3,4,skku_natural_back_gate,성균관대학교 자연과학캠퍼스 후문,300,762026624,코아김밥,음식점 > 분식,경기 수원시 장안구 율전동 419,경기 수원시 장안구 서부로 2095,126.969932,37.295950,031-278-0351,http://place.map.kakao.com/762026624,71.4
4,5,skku_natural_back_gate,성균관대학교 자연과학캠퍼스 후문,300,1138868621,김치로우주정복 수원율천점,"음식점 > 한식 > 찌개,전골 > 김치로우주정복",경기 수원시 장안구 율전동 419,경기 수원시 장안구 서부로 2095,126.969824,37.296121,NaN,http://place.map.kakao.com/1138868621,71.8


In [ ]:
!pip -q install selenium webdriver-manager tqdm
!apt-get update -qq
!apt-get install -y chromium-chromedriver

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  apparmor chromium-browser libfuse3-3 snapd squashfs-tools systemd-hwe-hwdb
  udev
Suggested packages:
  apparmor-profiles-extra apparmor-utils fuse3 zenity | kdialog
The following NEW packages will be installed:
  apparmor chromium-browser chromium-chromedriver libfuse3-3 snapd
  squashfs-tools systemd-hwe-hwdb udev
0 upgraded, 8 newly installed, 0 to remove and 5 not upgraded.
Need to get 35.2 MB of archives.
After this operation, 137 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 apparmor amd64 3.0.4-2ubuntu2.5 [599 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 squashfs-tools a

In [ ]:
import re
import random
import urllib.parse
from datetime import datetime

from tqdm import tqdm

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, StaleElementReferenceException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import shutil
from selenium.webdriver.chrome.service import Service

In [ ]:
candidate_path = DATA_DIR / "01_kakao_restaurant_candidates.csv"

df_candidates = pd.read_csv(candidate_path)

print(df_candidates.shape)
display(df_candidates[[
    "restaurant_temp_id",
    "name",
    "category_name",
    "address",
    "road_address",
    "longitude",
    "latitude",
    "distance_m"
]].head())

(45, 14)


,restaurant_temp_id,name,category_name,address,road_address,longitude,latitude,distance_m
0,1,낭만가득한20세기포장마차 수원성대본점,음식점 > 술집 > 실내포장마차,경기 수원시 장안구 율전동 419,경기 수원시 장안구 서부로 2095,126.969930,37.296044,66.2
1,2,소리산전집,음식점 > 한식,경기 수원시 장안구 율전동 433-79,경기 수원시 장안구 서부로 2107,126.970242,37.296854,68.2
2,3,두루정 수원성대점,음식점 > 한식,경기 수원시 장안구 율전동 433-79,경기 수원시 장안구 서부로 2107,126.970118,37.296795,68.7
3,4,코아김밥,음식점 > 분식,경기 수원시 장안구 율전동 419,경기 수원시 장안구 서부로 2095,126.969932,37.295950,71.4
4,5,김치로우주정복 수원율천점,"음식점 > 한식 > 찌개,전골 > 김치로우주정복",경기 수원시 장안구 율전동 419,경기 수원시 장안구 서부로 2095,126.969824,37.296121,71.8


In [ ]:
!pkill -f chrome || true
!pkill -f chromedriver || true

^C
^C


In [ ]:
!apt-get update -qq
!apt-get install -y wget gnupg unzip

!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
gnupg is already the newest version (2.2.27-3ubuntu2.5).
gnupg set to manually installed.
unzip is already the newest version (6.0-26ubuntu3.2).
wget is already the newest version (1.21.2-2ubuntu1.1).
0 upgraded, 0 newly installed, 0 to remove and 5 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'google-chrome-stable' instead of './google-chrome-stable_current_amd64.deb'
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0
  libatk1.0-data libatspi2.0-0 libvulkan1 libxcomposite1 libxtst6
  mesa-vulkan-drivers session-migration
The following NEW packages wi

In [ ]:
!google-chrome --version

Google Chrome 148.0.7778.178 


In [ ]:
!pip -q install -U selenium

In [ ]:
import selenium
print(selenium.__version__)

4.44.0


In [ ]:
def create_driver():
    options = Options()

    options.binary_location = "/usr/bin/google-chrome"

    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-software-rasterizer")
    options.add_argument("--window-size=1400,1000")
    options.add_argument("--lang=ko-KR")
    options.add_argument("--user-data-dir=/tmp/chrome-user-data")
    options.add_argument("--remote-debugging-port=9222")

    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(30)

    return driver

In [ ]:
driver = create_driver()

driver.get("https://www.naver.com")
print(driver.title)

driver.quit()

NAVER


In [ ]:
def unique_keep_order(items):
    seen = set()
    result = []

    for item in items:
        item = normalize_text(item)
        if item and item not in seen:
            seen.add(item)
            result.append(item)

    return result

def make_base_store_names(name):
    name = normalize_text(name)

    # 지점명 제거 패턴
    suffix_patterns = [
        r"수원성대본점$",
        r"수원성대점$",
        r"성대본점$",
        r"성대점$",
        r"수원율전점$",
        r"율전점$",
        r"수원점$",
        r"본점$",
        r"\d+호점$",
    ]

    names = [name]

    base = name
    for pattern in suffix_patterns:
        base = re.sub(pattern, "", base).strip()

    names.append(base)

    # 괄호 제거 버전
    no_paren = re.sub(r"\([^)]*\)", "", name).strip()
    names.append(no_paren)

    no_paren_base = no_paren
    for pattern in suffix_patterns:
        no_paren_base = re.sub(pattern, "", no_paren_base).strip()

    names.append(no_paren_base)

    return unique_keep_order(names)

def make_naver_search_queries(row):
    name = normalize_text(row["name"])
    address = normalize_text(row.get("road_address") or row.get("address") or "")

    base_names = make_base_store_names(name)

    queries = []

    # 1순위: 지점명 제거된 이름 단독 검색
    for store_name in base_names:
        queries.append(store_name)

    # 2순위: 위치 키워드 조합
    for store_name in base_names:
        queries.extend([
            f"{store_name} 수원 성대",
            f"{store_name} 수원성대",
            f"{store_name} 율전동",
            f"{store_name} 성균관대",
        ])

    # 3순위: 전체 주소는 마지막 fallback
    queries.append(f"{name} {address}")

    return unique_keep_order(queries)

In [ ]:
row = df_candidates.iloc[0]

print("원본 이름:", row["name"])
print("검색 쿼리 후보:")
for q in make_naver_search_queries(row):
    print("-", q)

원본 이름: 낭만가득한20세기포장마차 수원성대본점
검색 쿼리 후보:
- 낭만가득한20세기포장마차 수원성대본점
- 낭만가득한20세기포장마차
- 낭만가득한20세기포장마차 수원성대본점 수원 성대
- 낭만가득한20세기포장마차 수원성대본점 수원성대
- 낭만가득한20세기포장마차 수원성대본점 율전동
- 낭만가득한20세기포장마차 수원성대본점 성균관대
- 낭만가득한20세기포장마차 수원 성대
- 낭만가득한20세기포장마차 수원성대
- 낭만가득한20세기포장마차 율전동
- 낭만가득한20세기포장마차 성균관대
- 낭만가득한20세기포장마차 수원성대본점 경기 수원시 장안구 서부로 2095


In [ ]:
def get_all_visible_text(driver):
    texts = []

    try:
        switch_to_default(driver)
        texts.append(driver.find_element(By.TAG_NAME, "body").text)
    except Exception:
        pass

    try:
        switch_to_default(driver)
        frames = driver.find_elements(By.TAG_NAME, "iframe")

        for frame in frames:
            try:
                switch_to_default(driver)
                driver.switch_to.frame(frame)
                texts.append(driver.find_element(By.TAG_NAME, "body").text)
            except Exception:
                continue
    finally:
        switch_to_default(driver)

    return normalize_text(" ".join(texts))
def is_no_result_page(text):
    no_result_phrases = [
        "검색 결과가 없습니다",
        "검색결과가 없습니다",
        "찾으시는 검색 결과가 없다면",
        "단어의 철자가 정확한지 확인해 보세요",
    ]

    return any(phrase in text for phrase in no_result_phrases)

def parse_visitor_review_count(text):
    text = normalize_text(text)

    patterns = [
        r"방문자\s*리뷰\s*([\d,]+)",
        r"방문자리뷰\s*([\d,]+)",
        r"방문자\s*([\d,]+)",
        r"리뷰\s*([\d,]+)",
    ]

    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return int(match.group(1).replace(",", ""))

    return 0
def click_first_result_in_current_context(driver, expected_name):
    short_name = normalize_text(expected_name)[:6]

    xpath_candidates = [
        "//a[contains(@href, '/place/') or contains(@href, 'entry/place')]",
        f"//a[contains(normalize-space(), '{short_name}')]",
        f"//button[contains(normalize-space(), '{short_name}')]",
        f"//*[@role='button' and contains(normalize-space(), '{short_name}')]",
        f"//*[contains(normalize-space(), '{short_name}')]",
    ]

    for xpath in xpath_candidates:
        elements = driver.find_elements(By.XPATH, xpath)

        for element in elements[:10]:
            try:
                if element.is_displayed():
                    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", element)
                    sleep_jitter(0.3, 0.7)
                    driver.execute_script("arguments[0].click();", element)
                    sleep_jitter(2.0, 3.0)
                    return True
            except Exception:
                continue

    return False


def click_first_search_result_flexible(driver, expected_name):
    """
    1. 메인 페이지에서 먼저 클릭 시도
    2. 실패하면 searchIframe 안에서 클릭 시도
    """
    switch_to_default(driver)

    if click_first_result_in_current_context(driver, expected_name):
        return True

    if switch_to_iframe_by_keyword(driver, "searchIframe", timeout=3):
        if click_first_result_in_current_context(driver, expected_name):
            switch_to_default(driver)
            return True

    switch_to_default(driver)
    return False

In [ ]:
def is_access_limited_page(text):
    text = normalize_text(text)

    blocked_phrases = [
        "서비스 이용이 제한되었습니다",
        "과도한 접근 요청으로 서비스 이용이 제한되었습니다",
        "잠시 후 다시 시도해주세요",
        "이용이 제한되었습니다",
    ]

    return any(phrase in text for phrase in blocked_phrases)

In [ ]:
NAVER_PLACE_SEARCH_URL = "https://pcmap.place.naver.com/restaurant/list"

def make_naver_place_search_url(query):
    return NAVER_PLACE_SEARCH_URL + "?query=" + urllib.parse.quote(query)

def extract_naver_place_id_from_url(url):
    if not url:
        return None

    patterns = [
        r"/restaurant/(\d+)",
        r"/place/(\d+)",
        r"place/(\d+)",
        r"id=(\d+)",
    ]

    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)

    return None

def click_first_place_result_pcmap(driver, expected_name):
    short_name = normalize_text(expected_name)[:6]

    xpath_candidates = [
        "//a[contains(@href, '/restaurant/')]",
        "//a[contains(@href, '/place/')]",
        f"//a[contains(normalize-space(), '{short_name}')]",
        f"//*[contains(normalize-space(), '{short_name}')]",
    ]

    for xpath in xpath_candidates:
        elements = driver.find_elements(By.XPATH, xpath)

        for element in elements[:20]:
            try:
                if element.is_displayed():
                    href = element.get_attribute("href")
                    text = normalize_text(element.text)

                    driver.execute_script(
                        "arguments[0].scrollIntoView({block: 'center'});",
                        element
                    )
                    sleep_jitter(0.5, 1.0)
                    driver.execute_script("arguments[0].click();", element)
                    sleep_jitter(2.5, 3.5)

                    return True, href, text

            except Exception:
                continue

    return False, None, None


def sleep_jitter(a=1.0, b=2.0):
    time.sleep(random.uniform(a, b))


def normalize_text(text):
    return re.sub(r"\s+", " ", str(text or "")).strip()


def switch_to_default(driver):
    driver.switch_to.default_content()


def switch_to_iframe_by_keyword(driver, keyword, timeout=10):
    """
    네이버 지도는 searchIframe, entryIframe을 사용한다.
    iframe id/name/src 중 keyword가 포함된 프레임으로 전환한다.
    """
    end_time = time.time() + timeout

    while time.time() < end_time:
        switch_to_default(driver)

        frames = driver.find_elements(By.TAG_NAME, "iframe")

        for frame in frames:
            frame_id = frame.get_attribute("id") or ""
            frame_name = frame.get_attribute("name") or ""
            frame_src = frame.get_attribute("src") or ""

            if keyword in frame_id or keyword in frame_name or keyword in frame_src:
                driver.switch_to.frame(frame)
                return True

        time.sleep(0.5)

    return False


def get_iframe_src_by_keyword(driver, keyword):
    switch_to_default(driver)

    frames = driver.find_elements(By.TAG_NAME, "iframe")

    for frame in frames:
        frame_id = frame.get_attribute("id") or ""
        frame_name = frame.get_attribute("name") or ""
        frame_src = frame.get_attribute("src") or ""

        if keyword in frame_id or keyword in frame_name or keyword in frame_src:
            return frame_src

    return None


def save_debug_artifacts(driver, name_prefix):
    """
    실패했을 때 현재 화면/HTML을 저장해서 원인 확인용으로 사용.
    """
    debug_dir = DATA_DIR / "debug"
    debug_dir.mkdir(parents=True, exist_ok=True)

    safe_name = re.sub(r"[^가-힣a-zA-Z0-9_-]+", "_", str(name_prefix))[:40]

    png_path = debug_dir / f"{safe_name}.png"
    html_path = debug_dir / f"{safe_name}.html"

    try:
        driver.save_screenshot(str(png_path))
    except Exception:
        pass

    try:
        html_path.write_text(driver.page_source, encoding="utf-8")
    except Exception:
        pass

    return str(png_path), str(html_path)


def map_one_restaurant_to_naver_v3(driver, row):
    name = normalize_text(row["name"])
    queries = make_naver_search_queries(row)

    base_result = {
        "restaurant_temp_id": row["restaurant_temp_id"],
        "name": name,
        "kakao_address": row.get("address"),
        "kakao_road_address": row.get("road_address"),
        "kakao_longitude": row.get("longitude"),
        "kakao_latitude": row.get("latitude"),
        "kakao_place_id": row.get("kakao_place_id"),
        "kakao_place_url": row.get("kakao_place_url"),
        "naver_search_query": None,
        "naver_search_url": None,
        "naver_matched": False,
        "naver_place_id": None,
        "naver_place_url": None,
        "visitor_review_count": 0,
        "naver_page_text_sample": "",
        "attempted_queries": "",
        "debug_png": None,
        "debug_html": None,
        "error": None,
    }

    attempted = []
    last_result = base_result.copy()

    for query in queries:
        search_url = make_naver_place_search_url(query)

        result = base_result.copy()
        result["naver_search_query"] = query
        result["naver_search_url"] = search_url

        try:
            driver.get(search_url)
            sleep_jitter(4.0, 5.5)

            before_text = get_all_visible_text(driver)
            no_result = is_no_result_page(before_text)
            count_before_click = parse_visitor_review_count(before_text)
            if is_access_limited_page(before_text):
              result["error"] = "access limited"
              result["naver_page_text_sample"] = before_text[:500]
              result["attempted_queries"] = " || ".join(attempted)
              return result

            clicked, href, clicked_text = click_first_place_result_pcmap(driver, name)
            sleep_jitter(2.0, 3.0)

            after_text = get_all_visible_text(driver)
            count_after_click = parse_visitor_review_count(after_text)

            if is_access_limited_page(after_text):
              result["error"] = "access limited after click"
              result["naver_page_text_sample"] = after_text[:500]
              result["attempted_queries"] = " || ".join(attempted)
              return result

            current_url = driver.current_url
            naver_place_url = current_url

            place_id = (
                extract_naver_place_id_from_url(current_url)
                or extract_naver_place_id_from_url(href)
            )

            visitor_review_count = max(count_before_click, count_after_click)

            attempted.append(
                f"{query} | no_result={no_result} | clicked={clicked} "
                f"| count_before={count_before_click} | count_after={count_after_click} "
                f"| place_id={place_id} | clicked_text={clicked_text}"
            )

            if clicked or place_id or visitor_review_count > 0:
                result["naver_matched"] = True
                result["naver_place_id"] = place_id
                result["naver_place_url"] = naver_place_url
                result["visitor_review_count"] = visitor_review_count
                result["naver_page_text_sample"] = after_text[:500] if after_text else before_text[:500]
                result["attempted_queries"] = " || ".join(attempted)
                result["error"] = None
                return result

            result["error"] = "pcmap result parse failed"
            result["naver_page_text_sample"] = before_text[:500]
            last_result = result

        except Exception as e:
            attempted.append(f"{query} | exception={repr(e)}")
            result["error"] = repr(e)
            last_result = result

    png, html = save_debug_artifacts(driver, f"v3_failed_{name}")

    last_result["debug_png"] = png
    last_result["debug_html"] = html
    last_result["attempted_queries"] = " || ".join(attempted)

    return last_result

In [ ]:
def get_iframe_debug_info(driver):
    switch_to_default(driver)

    frames = driver.find_elements(By.TAG_NAME, "iframe")
    infos = []

    for i, frame in enumerate(frames):
        try:
            infos.append({
                "index": i,
                "id": frame.get_attribute("id"),
                "name": frame.get_attribute("name"),
                "src": frame.get_attribute("src"),
            })
        except Exception as e:
            infos.append({
                "index": i,
                "id": None,
                "name": None,
                "src": None,
                "error": repr(e),
            })

    return infos


def compact_iframe_info(infos):
    parts = []
    for info in infos:
        parts.append(
            f"[{info.get('index')}] "
            f"id={info.get('id')} "
            f"name={info.get('name')} "
            f"src={str(info.get('src'))[:120]}"
        )
    return " | ".join(parts)

In [ ]:
def make_naver_visitor_review_url(place_id):
    return f"https://pcmap.place.naver.com/restaurant/{place_id}/review/visitor"


def extract_review_like_texts_from_visible_text(text):
    text = normalize_text(text)

    # 너무 거친 1차 테스트용.
    # 실제 리뷰 추출 단계에서는 DOM selector 기반으로 더 정교화해야 함.
    lines = re.split(r"(?<=[.!?。])\s+|\n+", text)

    review_candidates = []

    bad_phrases = [
        "방문자 리뷰",
        "블로그 리뷰",
        "리뷰",
        "홈",
        "메뉴",
        "사진",
        "지도",
        "주변",
        "저장",
        "공유",
        "길찾기",
        "전화",
        "영업시간",
        "서비스 이용이 제한되었습니다",
        "과도한 접근 요청",
    ]

    for line in lines:
        line = normalize_text(line)

        if len(line) < 8:
            continue

        if len(line) > 500:
            continue

        if any(bad in line for bad in bad_phrases):
            continue

        # 한글이 거의 없는 텍스트 제거
        hangul_count = sum("가" <= ch <= "힣" for ch in line)
        if hangul_count < 5:
            continue

        review_candidates.append(line)

    return list(dict.fromkeys(review_candidates))


def test_one_place_reviews(driver, place_id, max_scroll=5):
    review_url = make_naver_visitor_review_url(place_id)

    result = {
        "place_id": place_id,
        "review_url": review_url,
        "access_limited": False,
        "visitor_review_count": 0,
        "sample_reviews": [],
        "page_text_sample": "",
        "error": None,
    }

    try:
        driver.get(review_url)
        sleep_jitter(4.0, 5.0)

        for _ in range(max_scroll):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            sleep_jitter(1.0, 1.5)

        text = get_all_visible_text(driver)

        result["page_text_sample"] = text[:500]
        result["access_limited"] = is_access_limited_page(text)

        if result["access_limited"]:
            result["error"] = "access limited"
            return result

        result["visitor_review_count"] = parse_visitor_review_count(text)
        result["sample_reviews"] = extract_review_like_texts_from_visible_text(text)[:10]

        return result

    except Exception as e:
        result["error"] = repr(e)
        return result

In [ ]:
test_place_id = "1385583026"

driver = create_driver()

try:
    result = test_one_place_reviews(driver, test_place_id)

    print("place_id:", result["place_id"])
    print("review_url:", result["review_url"])
    print("access_limited:", result["access_limited"])
    print("visitor_review_count:", result["visitor_review_count"])
    print("error:", result["error"])
    print("sample_reviews:")
    for r in result["sample_reviews"]:
        print("-", r)

finally:
    driver.quit()

place_id: 1385583026
review_url: https://pcmap.place.naver.com/restaurant/1385583026/review/visitor
access_limited: True
visitor_review_count: 0
error: access limited
sample_reviews:


In [ ]:
driver = create_driver()

try:
    row = df_candidates.iloc[0]
    mapped = map_one_restaurant_to_naver_v3(driver, row)

    print("name:", mapped["name"])
    print("matched:", mapped["naver_matched"])
    print("reviews:", mapped["visitor_review_count"])
    print("place_id:", mapped["naver_place_id"])
    print("place_url:", mapped["naver_place_url"])
    print("final_query:", mapped["naver_search_query"])
    print("error:", mapped["error"])
    print("attempted_queries:")
    print(mapped["attempted_queries"])

finally:
    driver.quit()

name: 낭만가득한20세기포장마차 수원성대본점
matched: True
reviews: 0
place_id: None
place_url: https://pcmap.place.naver.com/place/list?query=%EB%82%AD%EB%A7%8C%EA%B0%80%EB%93%9D%ED%95%9C20%EC%84%B8%EA%B8%B0%ED%8F%AC%EC%9E%A5%EB%A7%88%EC%B0%A8%20%EC%88%98%EC%9B%90%EC%84%B1%EB%8C%80%EB%B3%B8%EC%A0%90&noredirect=true&redirectCounter=1
final_query: 낭만가득한20세기포장마차 수원성대본점
error: None
attempted_queries:
낭만가득한20세기포장마차 수원성대본점 | no_result=False | clicked=True | count_before=0 | count_after=0 | place_id=None | clicked_text=이전 페이지 플레이스 페이지 닫기 서비스 이용이 제한되었습니다. 과도한 접근 요청으로 서비스 이용이 제한되었습니다. 잠시 후 다시 시도해주세요. IP: 34.139.16.127(2026.05.28 00:27:25) 이전화면새로고침


In [ ]:
driver = create_driver()

test_rows = []

try:
    for _, row in df_candidates.head(3).iterrows():
        mapped = map_one_restaurant_to_naver_v2(driver, row)
        test_rows.append(mapped)

        print(
            mapped["restaurant_temp_id"],
            mapped["name"],
            "query=", mapped["naver_search_query"],
            "matched=", mapped["naver_matched"],
            "reviews=", mapped["visitor_review_count"],
            "error=", mapped["error"]
        )

        sleep_jitter(2.0, 3.0)
finally:
    driver.quit()

df_test_mapped_v2 = pd.DataFrame(test_rows)

display(df_test_mapped_v2[[
    "restaurant_temp_id",
    "name",
    "naver_search_query",
    "naver_matched",
    "visitor_review_count",
    "naver_place_url",
    "error",
    "naver_page_text_sample",
    "debug_png"
]])

1 낭만가득한20세기포장마차 수원성대본점 query= 낭만가득한20세기포장마차 수원성대본점 경기 수원시 장안구 서부로 2095 matched= False reviews= 0 error= no result
2 소리산전집 query= 소리산전집 경기 수원시 장안구 서부로 2107 matched= False reviews= 0 error= no result
3 두루정 수원성대점 query= 두루정 수원성대점 경기 수원시 장안구 서부로 2107 matched= False reviews= 0 error= no result


,restaurant_temp_id,name,naver_search_query,naver_matched,visitor_review_count,naver_place_url,error,naver_page_text_sample,debug_png
0,1,낭만가득한20세기포장마차 수원성대본점,낭만가득한20세기포장마차 수원성대본점 경기 수원시 장안구 서부로 2095,False,0,None,no result,,/content/drive/MyDrive/Crawler/data/debug/v2_f...
1,2,소리산전집,소리산전집 경기 수원시 장안구 서부로 2107,False,0,None,no result,,/content/drive/MyDrive/Crawler/data/debug/v2_f...
2,3,두루정 수원성대점,두루정 수원성대점 경기 수원시 장안구 서부로 2107,False,0,None,no result,,/content/drive/MyDrive/Crawler/data/debug/v2_f...


In [ ]:
partial_path = DATA_DIR / "03_naver_mapped_restaurants_partial_debug.csv"
df_partial = pd.read_csv(partial_path)

In [ ]:
df_partial = pd.DataFrame(mapped_rows)

partial_path = DATA_DIR / "03_naver_mapped_restaurants_partial_debug.csv"
df_partial.to_csv(partial_path, index=False, encoding="utf-8-sig")

print("저장 완료:", partial_path)
print(df_partial.shape)

display(df_partial[[
    "restaurant_temp_id",
    "name",
    "naver_matched",
    "visitor_review_count",
    "error",
    "current_url",
    "page_title",
    "iframe_debug",
    "debug_png",
    "debug_html"
]].head(30))

저장 완료: /content/drive/MyDrive/Crawler/data/02_naver_mapped_restaurants_partial_debug.csv
(24, 20)


,restaurant_temp_id,name,naver_matched,visitor_review_count,error,current_url,page_title,iframe_debug,debug_png,debug_html
0,1,낭만가득한20세기포장마차 수원성대본점,False,0,entryIframe not found,https://map.naver.com/p/search/%EB%82%AD%EB%A7...,낭만가득한20세기포장마차 수원성대본점 경기 수원시 장안구 서부로 2095 검색 - ...,[0] id=gnb_my_lyr_iframe name=padding src= | [...,/content/drive/MyDrive/Crawler/data/debug/entr...,/content/drive/MyDrive/Crawler/data/debug/entr...
1,2,소리산전집,False,0,entryIframe not found,https://map.naver.com/p/search/%EC%86%8C%EB%A6...,소리산전집 경기 수원시 장안구 서부로 2107 검색 - 네이버지도,[0] id=gnb_my_lyr_iframe name=padding src= | [...,/content/drive/MyDrive/Crawler/data/debug/entr...,/content/drive/MyDrive/Crawler/data/debug/entr...
2,3,두루정 수원성대점,False,0,entryIframe not found,https://map.naver.com/p/search/%EB%91%90%EB%A3...,두루정 수원성대점 경기 수원시 장안구 서부로 2107 검색 - 네이버지도,[0] id=gnb_my_lyr_iframe name=padding src= | [...,/content/drive/MyDrive/Crawler/data/debug/entr...,/content/drive/MyDrive/Crawler/data/debug/entr...
3,4,코아김밥,False,0,entryIframe not found,https://map.naver.com/p/search/%EC%BD%94%EC%95...,코아김밥 경기 수원시 장안구 서부로 2095 검색 - 네이버지도,[0] id=gnb_my_lyr_iframe name=padding src= | [...,/content/drive/MyDrive/Crawler/data/debug/entr...,/content/drive/MyDrive/Crawler/data/debug/entr...
4,5,김치로우주정복 수원율천점,False,0,entryIframe not found,https://map.naver.com/p/search/%EA%B9%80%EC%B9...,김치로우주정복 수원율천점 경기 수원시 장안구 서부로 2095 검색 - 네이버지도,[0] id=gnb_my_lyr_iframe name=padding src= | [...,/content/drive/MyDrive/Crawler/data/debug/entr...,/content/drive/MyDrive/Crawler/data/debug/entr...
5,6,최고집 원조 수구레,False,0,entryIframe not found,https://map.naver.com/p/search/%EC%B5%9C%EA%B3...,최고집 원조 수구레 경기 수원시 장안구 서부로 2107 검색 - 네이버지도,[0] id=gnb_my_lyr_iframe name=padding src= | [...,/content/drive/MyDrive/Crawler/data/debug/entr...,/content/drive/MyDrive/Crawler/data/debug/entr...
6,7,돼지게티 율천점,False,0,entryIframe not found,https://map.naver.com/p/search/%EB%8F%BC%EC%A7...,돼지게티 율천점 경기 수원시 장안구 서부로 2095 검색 - 네이버지도,[0] id=gnb_my_lyr_iframe name=padding src= | [...,/content/drive/MyDrive/Crawler/data/debug/entr...,/content/drive/MyDrive/Crawler/data/debug/entr...
7,8,미,False,0,entryIframe not found,https://map.naver.com/p/search/%EB%AF%B8%20%EA...,미 경기 수원시 장안구 서부로 2107 검색 - 네이버지도,[0] id=gnb_my_lyr_iframe name=padding src= | [...,/content/drive/MyDrive/Crawler/data/debug/entr...,/content/drive/MyDrive/Crawler/data/debug/entr...
8,9,신전떡볶이 수원성대점,False,0,entryIframe not found,https://map.naver.com/p/search/%EC%8B%A0%EC%A0...,신전떡볶이 수원성대점 경기 수원시 장안구 서부로 2107 검색 - 네이버지도,[0] id=gnb_my_lyr_iframe name=padding src= | [...,/content/drive/MyDrive/Crawler/data/debug/entr...,/content/drive/MyDrive/Crawler/data/debug/entr...
9,10,중경마라탕 3호점,False,0,entryIframe not found,https://map.naver.com/p/search/%EC%A4%91%EA%B2...,중경마라탕 3호점 경기 수원시 장안구 서부로2106번길 10 검색 - 네이버지도,[0] id=gnb_my_lyr_iframe name=padding src= | [...,/content/drive/MyDrive/Crawler/data/debug/entr...,/content/drive/MyDrive/Crawler/data/debug/entr...


In [ ]:
print("전체 시도:", len(df_partial))
print("매칭 성공:", df_partial["naver_matched"].sum())
print("리뷰 수 추출 성공:", (df_partial["visitor_review_count"] > 0).sum())

display(
    df_partial.groupby(["naver_matched", "error"])
    .size()
    .reset_index(name="count")
)

전체 시도: 24
매칭 성공: 0
리뷰 수 추출 성공: 0


,naver_matched,error,count
0,False,entryIframe not found,24


In [ ]:
display(df_partial[[
    "restaurant_temp_id",
    "name",
    "current_url",
    "page_title",
    "iframe_debug",
    "debug_png",
    "debug_html"
]].head(10))

,restaurant_temp_id,name,current_url,page_title,iframe_debug,debug_png,debug_html
0,1,낭만가득한20세기포장마차 수원성대본점,https://map.naver.com/p/search/%EB%82%AD%EB%A7...,낭만가득한20세기포장마차 수원성대본점 경기 수원시 장안구 서부로 2095 검색 - ...,[0] id=gnb_my_lyr_iframe name=padding src= | [...,/content/drive/MyDrive/Crawler/data/debug/entr...,/content/drive/MyDrive/Crawler/data/debug/entr...
1,2,소리산전집,https://map.naver.com/p/search/%EC%86%8C%EB%A6...,소리산전집 경기 수원시 장안구 서부로 2107 검색 - 네이버지도,[0] id=gnb_my_lyr_iframe name=padding src= | [...,/content/drive/MyDrive/Crawler/data/debug/entr...,/content/drive/MyDrive/Crawler/data/debug/entr...
2,3,두루정 수원성대점,https://map.naver.com/p/search/%EB%91%90%EB%A3...,두루정 수원성대점 경기 수원시 장안구 서부로 2107 검색 - 네이버지도,[0] id=gnb_my_lyr_iframe name=padding src= | [...,/content/drive/MyDrive/Crawler/data/debug/entr...,/content/drive/MyDrive/Crawler/data/debug/entr...
3,4,코아김밥,https://map.naver.com/p/search/%EC%BD%94%EC%95...,코아김밥 경기 수원시 장안구 서부로 2095 검색 - 네이버지도,[0] id=gnb_my_lyr_iframe name=padding src= | [...,/content/drive/MyDrive/Crawler/data/debug/entr...,/content/drive/MyDrive/Crawler/data/debug/entr...
4,5,김치로우주정복 수원율천점,https://map.naver.com/p/search/%EA%B9%80%EC%B9...,김치로우주정복 수원율천점 경기 수원시 장안구 서부로 2095 검색 - 네이버지도,[0] id=gnb_my_lyr_iframe name=padding src= | [...,/content/drive/MyDrive/Crawler/data/debug/entr...,/content/drive/MyDrive/Crawler/data/debug/entr...
5,6,최고집 원조 수구레,https://map.naver.com/p/search/%EC%B5%9C%EA%B3...,최고집 원조 수구레 경기 수원시 장안구 서부로 2107 검색 - 네이버지도,[0] id=gnb_my_lyr_iframe name=padding src= | [...,/content/drive/MyDrive/Crawler/data/debug/entr...,/content/drive/MyDrive/Crawler/data/debug/entr...
6,7,돼지게티 율천점,https://map.naver.com/p/search/%EB%8F%BC%EC%A7...,돼지게티 율천점 경기 수원시 장안구 서부로 2095 검색 - 네이버지도,[0] id=gnb_my_lyr_iframe name=padding src= | [...,/content/drive/MyDrive/Crawler/data/debug/entr...,/content/drive/MyDrive/Crawler/data/debug/entr...
7,8,미,https://map.naver.com/p/search/%EB%AF%B8%20%EA...,미 경기 수원시 장안구 서부로 2107 검색 - 네이버지도,[0] id=gnb_my_lyr_iframe name=padding src= | [...,/content/drive/MyDrive/Crawler/data/debug/entr...,/content/drive/MyDrive/Crawler/data/debug/entr...
8,9,신전떡볶이 수원성대점,https://map.naver.com/p/search/%EC%8B%A0%EC%A0...,신전떡볶이 수원성대점 경기 수원시 장안구 서부로 2107 검색 - 네이버지도,[0] id=gnb_my_lyr_iframe name=padding src= | [...,/content/drive/MyDrive/Crawler/data/debug/entr...,/content/drive/MyDrive/Crawler/data/debug/entr...
9,10,중경마라탕 3호점,https://map.naver.com/p/search/%EC%A4%91%EA%B2...,중경마라탕 3호점 경기 수원시 장안구 서부로2106번길 10 검색 - 네이버지도,[0] id=gnb_my_lyr_iframe name=padding src= | [...,/content/drive/MyDrive/Crawler/data/debug/entr...,/content/drive/MyDrive/Crawler/data/debug/entr...


# **여기부터인사캠**
*

In [ ]:
# 인문사회과학캠퍼스 정문 기준 300m 이내 음식점 후보 수집
# 기존 kakao_key, BASE_DIR, DATA_DIR, haversine_m 재사용

CENTER_QUERY = "성균관대학교 인문사회과학캠퍼스 정문"
RADIUS_M = 300
CATEGORY_GROUP_CODE = "FD6"  # 음식점

headers = {
    "Authorization": f"KakaoAK {kakao_key.strip()}"
}

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)

# 1. 정문 중심 좌표 검색
url = "https://dapi.kakao.com/v2/local/search/keyword.json"
params = {
    "query": CENTER_QUERY,
    "size": 5
}

res = requests.get(url, headers=headers, params=params, timeout=10)
print("center search status:", res.status_code)

data = res.json()
docs = data.get("documents", [])

if not docs:
    raise RuntimeError("인문사회과학캠퍼스 정문 중심 좌표를 찾지 못했습니다.")

center = docs[0]

CENTER_LON = float(center["x"])
CENTER_LAT = float(center["y"])

print("중심 장소:", center.get("place_name"))
print("주소:", center.get("address_name"))
print("도로명:", center.get("road_address_name"))
print("중심 좌표:", CENTER_LON, CENTER_LAT)


# 2. 음식점 후보 수집 함수
def collect_kakao_restaurant_candidates_humanities(center_lon, center_lat, radius_m=300):
    url = "https://dapi.kakao.com/v2/local/search/category.json"
    rows = []

    for page in range(1, 46):
        params = {
            "category_group_code": CATEGORY_GROUP_CODE,
            "x": center_lon,
            "y": center_lat,
            "radius": radius_m,
            "sort": "distance",
            "page": page,
            "size": 15,
        }

        res = requests.get(url, headers=headers, params=params, timeout=10)
        print(f"page={page}, status={res.status_code}")

        if res.status_code != 200:
            print(res.text)
            break

        data = res.json()
        documents = data.get("documents", [])

        for d in documents:
            lon = float(d["x"])
            lat = float(d["y"])

            distance_m = haversine_m(center_lon, center_lat, lon, lat)

            if distance_m <= radius_m:
                rows.append({
                    "kakao_place_id": d.get("id"),
                    "name": d.get("place_name"),
                    "category_name": d.get("category_name"),
                    "address": d.get("address_name"),
                    "road_address": d.get("road_address_name"),
                    "longitude": lon,
                    "latitude": lat,
                    "phone": d.get("phone"),
                    "kakao_place_url": d.get("place_url"),
                    "distance_m": round(distance_m, 1),
                })

        if data.get("meta", {}).get("is_end"):
            break

        time.sleep(0.2)

    df = pd.DataFrame(rows)

    if df.empty:
        return df

    df = df.drop_duplicates(subset=["kakao_place_id"])
    df = df.sort_values(["distance_m", "name"]).reset_index(drop=True)

    df.insert(0, "restaurant_temp_id", range(1, len(df) + 1))
    df.insert(1, "campus_area", "skku_humanities_front_gate")
    df.insert(2, "center_query", CENTER_QUERY)
    df.insert(3, "radius_m", radius_m)

    return df


# 3. 수집 실행
df_humanities_candidates = collect_kakao_restaurant_candidates_humanities(
    center_lon=CENTER_LON,
    center_lat=CENTER_LAT,
    radius_m=RADIUS_M
)

print("수집된 음식점 후보 수:", len(df_humanities_candidates))

display_cols = [
    "restaurant_temp_id",
    "name",
    "category_name",
    "address",
    "road_address",
    "distance_m",
    "kakao_place_url",
]

display(df_humanities_candidates[display_cols].head(50))


# 4. 기존 입력 CSV와 같은 형식으로 저장
out_path = DATA_DIR / "01_kakao_restaurant_candidates_humanities.csv"
df_humanities_candidates.to_csv(out_path, index=False, encoding="utf-8-sig")

print("저장 완료:", out_path)

BASE_DIR: /content/drive/MyDrive/Crawler
DATA_DIR: /content/drive/MyDrive/Crawler/data
center search status: 200
중심 장소: 성균관대학교 인문사회과학캠퍼스 정문
주소: 서울 종로구 명륜3가 53
도로명: 서울 종로구 성균관로 25-1
중심 좌표: 126.996838384994 37.5850002309339
page=1, status=200
page=2, status=200
page=3, status=200
수집된 음식점 후보 수: 45


,restaurant_temp_id,name,category_name,address,road_address,distance_m,kakao_place_url
0,1,인생건어물맥주 대학로점,"음식점 > 술집 > 호프,요리주점 > 인생건어물맥주",서울 종로구 명륜2가 122,서울 종로구 성균관로 26,42.4,http://place.map.kakao.com/168487509
1,2,한솥도시락 성균관대앞점,음식점 > 도시락 > 한솥도시락,서울 종로구 명륜2가 122,서울 종로구 성균관로 26,42.7,http://place.map.kakao.com/11634881
2,3,맥덕스,음식점 > 양식,서울 종로구 명륜2가 135-3,서울 종로구 성균관로 25-9,43.6,http://place.map.kakao.com/22016023
3,4,명륜진사갈비 서울성균관대점,"음식점 > 한식 > 육류,고기 > 갈비 > 명륜진사갈비",서울 종로구 명륜3가 53,서울 종로구 성균관로 31,44.2,http://place.map.kakao.com/338935651
4,5,물결식당,음식점 > 퓨전요리 > 퓨전한식,서울 종로구 명륜2가 135-3,서울 종로구 성균관로 25-9,45.3,http://place.map.kakao.com/122821905
5,6,피자헤븐 종로성대점,음식점 > 양식 > 피자 > 피자헤븐,서울 종로구 명륜2가 135,서울 종로구 성균관로 21-2,50.3,http://place.map.kakao.com/25770908
6,7,맘스터치 피자앤치킨 종로성대점,음식점 > 양식 > 피자 > 맘스터치 피자앤치킨,서울 종로구 명륜2가 74-1,서울 종로구 성균관로 30,53.7,http://place.map.kakao.com/1869234515
7,8,명분,"음식점 > 간식 > 제과,베이커리",서울 종로구 명륜2가 130,서울 종로구 성균관로 26-4,54.7,http://place.map.kakao.com/483030266
8,9,집시포차 혜화점,음식점 > 술집 > 실내포장마차,서울 종로구 명륜2가 138-1,서울 종로구 성균관로 21,61.2,http://place.map.kakao.com/633176459
9,10,페르시안궁전,음식점 > 아시아음식 > 인도음식,서울 종로구 명륜2가 121-1,서울 종로구 성균관로6길 9,64.2,http://place.map.kakao.com/11520636


저장 완료: /content/drive/MyDrive/Crawler/data/01_kakao_restaurant_candidates_humanities.csv


# **여기서부터 통합좌표지정임**

In [15]:
print(kakao_key)

4a1bda9139cf2495c5f86aa27465d70c


In [16]:
# 경로 설정
N_DIR = DATA_DIR / "n"
H_DIR = DATA_DIR / "h"

# 자과캠 파일
n_status_path = N_DIR / "review_collection_status.csv"
n_candidates_path = N_DIR / "kakao_restaurant_candidates.csv"

# 인사캠 파일
h_status_path = H_DIR / "h_review_collection_status.csv"
h_candidates_path = H_DIR / "h_kakao_restaurant_candidates.csv"

# 출력 파일
out_path = DATA_DIR / "done_restaurant_locations.csv"

print("N status:", n_status_path)
print("N candidates:", n_candidates_path)
print("H status:", h_status_path)
print("H candidates:", h_candidates_path)
print("Output:", out_path)

N status: /content/drive/MyDrive/Crawler/data/n/review_collection_status.csv
N candidates: /content/drive/MyDrive/Crawler/data/n/kakao_restaurant_candidates.csv
H status: /content/drive/MyDrive/Crawler/data/h/h_review_collection_status.csv
H candidates: /content/drive/MyDrive/Crawler/data/h/h_kakao_restaurant_candidates.csv
Output: /content/drive/MyDrive/Crawler/data/done_restaurant_locations.csv


In [17]:
# 파일 존재 확인

paths = [
    n_status_path,
    n_candidates_path,
    h_status_path,
    h_candidates_path,
]

for p in paths:
    print(p, "->", p.exists())

missing_paths = [p for p in paths if not p.exists()]
if missing_paths:
    raise FileNotFoundError(f"파일이 없습니다: {missing_paths}")

/content/drive/MyDrive/Crawler/data/n/review_collection_status.csv -> True
/content/drive/MyDrive/Crawler/data/n/kakao_restaurant_candidates.csv -> True
/content/drive/MyDrive/Crawler/data/h/h_review_collection_status.csv -> True
/content/drive/MyDrive/Crawler/data/h/h_kakao_restaurant_candidates.csv -> True


In [18]:
# status + candidate 파일 읽기

df_n_status = pd.read_csv(n_status_path)
df_n_candidates = pd.read_csv(n_candidates_path)

df_h_status = pd.read_csv(h_status_path)
df_h_candidates = pd.read_csv(h_candidates_path)

print("자과캠 status:", df_n_status.shape)
print("자과캠 candidates:", df_n_candidates.shape)
print("인사캠 status:", df_h_status.shape)
print("인사캠 candidates:", df_h_candidates.shape)

print("\n자과캠 status columns:", df_n_status.columns.tolist())
print("자과캠 candidates columns:", df_n_candidates.columns.tolist())
print("\n인사캠 status columns:", df_h_status.columns.tolist())
print("인사캠 candidates columns:", df_h_candidates.columns.tolist())

자과캠 status: (45, 6)
자과캠 candidates: (45, 14)
인사캠 status: (45, 6)
인사캠 candidates: (45, 14)

자과캠 status columns: ['restaurant_external_id', 'restaurant_name', 'kakao_place_id', 'kakao_place_url', 'collected_review_count', 'status']
자과캠 candidates columns: ['restaurant_temp_id', 'campus_area', 'center_query', 'radius_m', 'kakao_place_id', 'name', 'category_name', 'address', 'road_address', 'longitude', 'latitude', 'phone', 'kakao_place_url', 'distance_m']

인사캠 status columns: ['restaurant_external_id', 'restaurant_name', 'kakao_place_id', 'kakao_place_url', 'collected_review_count', 'status']
인사캠 candidates columns: ['restaurant_temp_id', 'campus_area', 'center_query', 'radius_m', 'kakao_place_id', 'name', 'category_name', 'address', 'road_address', 'longitude', 'latitude', 'phone', 'kakao_place_url', 'distance_m']


In [19]:
# done 상태 식당만 추출해서 후보 파일의 위도/경도와 병합하는 함수

def build_done_location_df(df_status, df_candidates, campus_slug):
    required_status_cols = [
        "restaurant_external_id",
        "restaurant_name",
        "kakao_place_id",
        "status",
    ]

    required_candidate_cols = [
        "kakao_place_id",
        "latitude",
        "longitude",
    ]

    missing_status = [c for c in required_status_cols if c not in df_status.columns]
    missing_candidate = [c for c in required_candidate_cols if c not in df_candidates.columns]

    if missing_status:
        raise ValueError(f"{campus_slug} status 파일 필수 컬럼 누락: {missing_status}")

    if missing_candidate:
        raise ValueError(f"{campus_slug} candidates 파일 필수 컬럼 누락: {missing_candidate}")

    status_done = df_status.copy()

    status_done["status"] = status_done["status"].astype(str).str.strip().str.lower()
    status_done = status_done[status_done["status"] == "done"].copy()

    status_done["kakao_place_id"] = status_done["kakao_place_id"].astype(str)
    df_candidates = df_candidates.copy()
    df_candidates["kakao_place_id"] = df_candidates["kakao_place_id"].astype(str)

    candidate_location = (
        df_candidates[
            [
                "kakao_place_id",
                "latitude",
                "longitude",
            ]
        ]
        .drop_duplicates(subset=["kakao_place_id"])
        .copy()
    )

    merged = status_done.merge(
        candidate_location,
        on="kakao_place_id",
        how="left",
    )

    result = merged[
        [
            "restaurant_external_id",
            "restaurant_name",
            "latitude",
            "longitude",
        ]
    ].copy()

    result.insert(0, "campus_slug", campus_slug)

    return result

In [20]:
# 자과캠 + 인사캠 done 식당 위치 데이터 생성

df_n_locations = build_done_location_df(
    df_status=df_n_status,
    df_candidates=df_n_candidates,
    campus_slug="natural",
)

df_h_locations = build_done_location_df(
    df_status=df_h_status,
    df_candidates=df_h_candidates,
    campus_slug="humanities",
)

print("자과캠 done 식당 수:", len(df_n_locations))
print("인사캠 done 식당 수:", len(df_h_locations))

display(df_n_locations.head())
display(df_h_locations.head())

자과캠 done 식당 수: 40
인사캠 done 식당 수: 37


,campus_slug,restaurant_external_id,restaurant_name,latitude,longitude
0,natural,kakao_824736542,낭만가득한20세기포장마차 수원성대본점,37.296044,126.969930
1,natural,kakao_306868449,소리산전집,37.296854,126.970242
2,natural,kakao_1279255339,두루정 수원성대점,37.296795,126.970118
3,natural,kakao_762026624,코아김밥,37.295950,126.969932
4,natural,kakao_1676208114,최고집 원조 수구레,37.296801,126.970067


,campus_slug,restaurant_external_id,restaurant_name,latitude,longitude
0,humanities,kakao_168487509,인생건어물맥주 대학로점,37.585003,126.997319
1,humanities,kakao_11634881,한솥도시락 성균관대앞점,37.584999,126.997324
2,humanities,kakao_22016023,맥덕스,37.584635,126.997020
3,humanities,kakao_338935651,명륜진사갈비 서울성균관대점,37.585392,126.996753
4,humanities,kakao_122821905,물결식당,37.584619,126.997018


In [24]:
# 합본 생성
# 최종 요청 컬럼:
# 1열 restaurant_external_id
# 2열 restaurant_name
# 3열 campus_slug
# 4열 latitude
# 5열 longitude

df_locations_all = pd.concat(
    [df_n_locations, df_h_locations],
    ignore_index=True,
)

df_locations_all = df_locations_all[
    [
        "restaurant_external_id",
        "restaurant_name",
        "campus_slug",
        "latitude",
        "longitude",
    ]
].copy()

# 중복 제거
df_locations_all = df_locations_all.drop_duplicates(
    subset=["restaurant_external_id"]
).reset_index(drop=True)

out_path = DATA_DIR / "done_restaurant_locations.csv"
df_locations_all.to_csv(out_path, index=False, encoding="utf-8-sig")

print("저장 완료:", out_path)
print("합본 done 식당 수:", len(df_locations_all))

display(df_locations_all.head(100))

저장 완료: /content/drive/MyDrive/Crawler/data/done_restaurant_locations.csv
합본 done 식당 수: 77


,restaurant_external_id,restaurant_name,campus_slug,latitude,longitude
0,kakao_824736542,낭만가득한20세기포장마차 수원성대본점,natural,37.296044,126.969930
1,kakao_306868449,소리산전집,natural,37.296854,126.970242
2,kakao_1279255339,두루정 수원성대점,natural,37.296795,126.970118
3,kakao_762026624,코아김밥,natural,37.295950,126.969932
4,kakao_1676208114,최고집 원조 수구레,natural,37.296801,126.970067
...,...,...,...,...,...
72,kakao_9029832,명륜쭈꾸미,humanities,37.584080,126.997136
73,kakao_131183348,양궈푸마라탕 성균관대점,humanities,37.584098,126.997419
74,kakao_1283367577,퍼센트,humanities,37.584098,126.997419
75,kakao_19976659,동두천부대찌개,humanities,37.583979,126.997256


In [23]:
print(a)

KAKAO_REST_API_KEY=4a1bda9139cf2495c5f86aa27465d70c



In [25]:
# 검증

df_check = pd.read_csv(out_path)

print("shape:", df_check.shape)
print("columns:", df_check.columns.tolist())

expected_cols = [
    "restaurant_external_id",
    "restaurant_name",
    "campus_slug",
    "latitude",
    "longitude",
]

if df_check.columns.tolist() != expected_cols:
    raise RuntimeError("최종 컬럼 순서가 요구사항과 다릅니다.")

dup_count = df_check["restaurant_external_id"].duplicated().sum()
null_slug = df_check["campus_slug"].isna().sum()
null_lat = df_check["latitude"].isna().sum()
null_lng = df_check["longitude"].isna().sum()

print("restaurant_external_id 중복 수:", dup_count)
print("campus_slug 결측 수:", null_slug)
print("latitude 결측 수:", null_lat)
print("longitude 결측 수:", null_lng)

print("\ncampus_slug 분포:")
print(df_check["campus_slug"].value_counts(dropna=False))

if dup_count != 0:
    raise RuntimeError("restaurant_external_id 중복이 있습니다.")

if null_slug != 0:
    raise RuntimeError("campus_slug 결측이 있습니다.")

if null_lat != 0 or null_lng != 0:
    print("위도/경도 결측 식당:")
    display(df_check[df_check["latitude"].isna() | df_check["longitude"].isna()])
    raise RuntimeError("위도/경도 결측이 있습니다.")

valid_slugs = {"natural", "humanities"}
bad_slugs = set(df_check["campus_slug"].dropna().unique()) - valid_slugs

if bad_slugs:
    raise RuntimeError(f"허용되지 않은 campus_slug가 있습니다: {bad_slugs}")

print("\n검증 통과")

shape: (77, 5)
columns: ['restaurant_external_id', 'restaurant_name', 'campus_slug', 'latitude', 'longitude']
restaurant_external_id 중복 수: 0
campus_slug 결측 수: 0
latitude 결측 수: 0
longitude 결측 수: 0

campus_slug 분포:
campus_slug
natural       40
humanities    37
Name: count, dtype: int64

검증 통과
